# One shot Ver4

In [1]:

import random
from util.analysis_algs import *
from archive.Subclass_inference import *

def fix_seeds(seed: int = 3407) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

fix_seeds(3402)

In [2]:
# load the model for one shot
# --ripple dataset--
weight_dir = r"C:\Users\user\PycharmProjects\subclass_segformer\output_mix5_sb2L2AW_new64_aug10_noshuf_gain_drop02_f5_300\ripple_2\SubclassSegFormer_MiT-B0_ripple.pth"

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = torch.load(weight_dir, weights_only=False).to(device).eval()

In [3]:

# ================= 設定區域 =================
fields = 'barrel,sea,LNG,seabass_hmh,noon_jsj'.split(',')
root = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"

def check_data_integrity(fields, root):
    print(f"{'Field':<15} | {'Fold 1':<8} | {'Fold 2':<8} | {'Fold 3':<8} | {'Fold 4':<8} | {'Total':<8}")
    print("-" * 75)

    grand_total = 0

    # 用來存資料給 Pandas 顯示 (選用)
    data = []

    for field in fields:
        row_counts = []
        field_total = 0

        for j in range(1, 5): # Fold 1 ~ 4
            # 組合路徑: organized_ripple_4fold/barrel/fold_1/images/*.png
            fold_path = os.path.join(root, field, f'fold_{j}', 'images')

            # 計算該資料夾下的 png 數量
            # 如果您的副檔名可能是 jpg，請自行調整或用 *.*
            images = glob.glob(os.path.join(fold_path, '*.png'))
            count = len(images)

            row_counts.append(count)
            field_total += count

            # 檢查是否有缺 (預期 125)
            if count != 125:
                # 標記異常
                row_counts[-1] = f"{count} (!)"

        grand_total += field_total

        # 顯示該場域的統計
        print(f"{field:<15} | {str(row_counts[0]):<8} | {str(row_counts[1]):<8} | {str(row_counts[2]):<8} | {str(row_counts[3]):<8} | {field_total:<8}")

    print("-" * 75)
    print(f"Grand Total: {grand_total} (Expected: 2500)")
    print(f"Missing: {2500 - grand_total}")

if __name__ == "__main__":
    check_data_integrity(fields, root)

Field           | Fold 1   | Fold 2   | Fold 3   | Fold 4   | Total   
---------------------------------------------------------------------------
barrel          | 125      | 125      | 125      | 125      | 500     
sea             | 125      | 121 (!)  | 125      | 125      | 496     
LNG             | 125      | 120 (!)  | 125      | 125      | 495     
seabass_hmh     | 124 (!)  | 121 (!)  | 125      | 125      | 495     
noon_jsj        | 125      | 125      | 125      | 125      | 500     
---------------------------------------------------------------------------
Grand Total: 2486 (Expected: 2500)
Missing: 14


In [4]:
# ================= 1. 資料準備 =================
fields = 'barrel,sea,LNG,seabass_hmh,noon_jsj'.split(',')
root = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"
train_imgs = get_ripple_data_paths(fields, root, k=-1, split_flag='training')
val_imgs   = get_ripple_data_paths(fields, root, k=-1, split_flag='validation')
training_data = train_imgs + val_imgs
print(f"Total Database Size: {len(training_data)}")

# # ================= 2. 執行特徵提取 (包含負樣本) =================
totalsubclass = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("--- 步驟 1：開始提取 SegFormer 特徵 (正樣本) 與 背景庫 (負樣本) ---")

r_basis, r_center, all_features_tensor, all_context_list, background_bank = get_segformer_features_and_context(
    model, training_data, totalsubclass, device, ver='total_points'
)
print("--- 步驟 1：提取完成 ---")

# ================= 3. 儲存結果 =================
output_dir = "../archive/prompt"
os.makedirs(output_dir, exist_ok=True)

print("正在儲存檔案...")
torch.save(r_basis, os.path.join(output_dir, "r_basis.pth"))
torch.save(r_center, os.path.join(output_dir, "r_center.pth"))
torch.save(all_features_tensor, os.path.join(output_dir, "all_features_tensor.pth"))
torch.save(all_context_list, os.path.join(output_dir, "all_context_list.pth"))
torch.save(background_bank, os.path.join(output_dir, "background_bank.pth"))

Total Database Size: 2475
--- 步驟 1：開始提取 SegFormer 特徵 (正樣本) 與 背景庫 (負樣本) ---


Extracting SegFormer Features & Context: 100%|██████████| 2475/2475 [03:21<00:00, 12.31it/s]


SegFormer 特徵提取完成 [shape: torch.Size([4118667, 64])]
上下文資訊 (Context) 筆數: 4118667

=== 背景庫 (Negative Bank) 統計 ===
Field ID   | Vectors Count  
------------------------------
A          | 49,463         
B          | 49,489         
C          | 49,081         
D          | 44,635         
E          | 39,855         
------------------------------
Total      | 232,523        

--- 步驟 1：提取完成 ---
正在儲存檔案...


In [5]:
r_basis = torch.load('../r_basis.pth', map_location='cpu')
r_center = torch.load('../r_center.pth', map_location='cpu')
all_features_tensor = torch.load('../archive/prompt/all_features_tensor.pth', map_location='cpu')
all_context_list = torch.load('../archive/prompt/all_context_list.pth', map_location='cpu')
background_bank = torch.load('../archive/prompt/background_bank.pth', map_location='cpu')

# (將結果轉換為 Numpy 以便後續聚類)
all_features_np = all_features_tensor.cpu().numpy()
print(f"已轉換為 Numpy 陣列，準備進行聚類。 Shape: {all_features_np.shape}")

C:\Users\user\AppData\Local\Temp\ipykernel_33288\3029570839.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  r_basis = torch.load('prompt/r_basis.pth', map_location='cpu'

已轉換為 Numpy 陣列，準備進行聚類。 Shape: (4118667, 64)


C:\Users\user\AppData\Local\Temp\ipykernel_33288\3029570839.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  background_bank = torch.load('prompt/background_bank.pth', ma

## 分群

In [7]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances

# --- 假設接續你的程式碼 ---
# all_features_np = ... (你的 N x 64 特徵矩陣)

# ================= 0. 資料前處理 =================
print("--- 步驟 0: 特徵正規化 ---")
# 雖然 SegFormer 輸出的特徵可能已經有經過 LayerNorm，但為了計算歐式距離準確，建議做 L2 Normalize
# 這樣歐式距離就會等價於 Cosine Similarity 的排序
all_features_norm = normalize(all_features_np, norm='l2', axis=1)

# ================= 1. K 值自動化 (Elbow / Silhouette) =================
print("--- 步驟 1: 自動尋找最佳 K 值 ---")

# 設定搜尋範圍
range_n_clusters = [8, 12, 16, 20, 24, 32]
best_k = 16  # 預設值，若計算失敗則回退
best_score = -1
scores = []

# 為了加速，隨機取樣 10,000 點來算 Silhouette (全量跑會非常久)
sample_size = 10000
if all_features_norm.shape[0] > sample_size:
    indices = np.random.choice(all_features_norm.shape[0], sample_size, replace=False)
    sample_data = all_features_norm[indices]
else:
    sample_data = all_features_norm

for n_clusters in range_n_clusters:
    # 這裡用 KMeans，也可以考慮 MiniBatchKMeans 加速
    clusterer = KMeans(n_clusters=n_clusters, random_state=3407, n_init='auto')
    cluster_labels = clusterer.fit_predict(sample_data)

    # 計算輪廓係數 (-1 到 1，越接近 1 越好)
    silhouette_avg = silhouette_score(sample_data, cluster_labels)
    scores.append(silhouette_avg)
    print(f"For n_clusters = {n_clusters}, The average silhouette_score is : {silhouette_avg:.4f}")

    if silhouette_avg > best_score:
        best_score = silhouette_avg
        best_k = n_clusters

print(f"===> 最佳 K 值為: {best_k} (Score: {best_score:.4f})")
# 如果你想強制固定 K=16，可以在這裡把 best_k 改回 16
# best_k = 16

# ================= 2. 執行最終 K-Means =================
print(f"--- 步驟 2: 使用 K={best_k} 對「全量數據」進行分群 ---")
kmeans = KMeans(n_clusters=best_k, random_state=3407, n_init=10)
labels = kmeans.fit_predict(all_features_norm)

# ================= 3. Medoid (代表點) 與 4. Prior (先驗機率) =================
print("--- 步驟 3 & 4: 計算 Medoid 與 Prior ---")

# 準備容器
visual_bases_medoids = []
priors = []
cluster_indices_collection = [] # 紀錄每一群對應的原始 index (之後若要存圖可能會用到)

total_samples = all_features_norm.shape[0]

for i in range(best_k):
    # 3.1 找出屬於這一群的所有點的索引
    indices = np.where(labels == i)[0]
    cluster_points = all_features_norm[indices]

    # 3.2 計算 Prior (該群數量 / 總數量)
    prob = len(indices) / total_samples
    priors.append(prob)

    # 3.3 計算 Medoid (找出距離質心最近的真實點)
    # 取得這一群的質心 (虛擬點)
    centroid = kmeans.cluster_centers_[i].reshape(1, -1)

    # 計算該群所有點到質心的距離
    # 注意：雖然可以用 cosine similarity，但因為前面做了 L2 normalize，歐式距離最小 = Cosine 距離最小
    dists = euclidean_distances(cluster_points, centroid)

    # 找出最小距離的 index (這是相對於 indices 的 index)
    min_dist_idx = np.argmin(dists)

    # 取得真實的 Medoid 特徵向量
    medoid_vector = cluster_points[min_dist_idx]
    visual_bases_medoids.append(medoid_vector)

    # (可選) 紀錄這是原始資料中的第幾筆，方便之後回去找原始圖片
    original_idx = indices[min_dist_idx]

    print(f"Cluster {i:02d}: Count={len(indices)}, Prior={prob:.4f}, Medoid Original Index={original_idx}")

# 轉換為 Tensor
visual_bases_tensor = torch.tensor(np.array(visual_bases_medoids)).float()
priors_tensor = torch.tensor(np.array(priors)).float()

print("\n--- 結果摘要 ---")
print(f"Visual Bases Shape: {visual_bases_tensor.shape}") # 預期 (K, 64)
print(f"Priors Shape: {priors_tensor.shape}")             # 預期 (K, )

# ================= 5. 儲存結果 =================
output_dir = "../archive/prompt"  # 假設存到 prompt 資料夾
import os
os.makedirs(output_dir, exist_ok=True)

torch.save(visual_bases_tensor, os.path.join(output_dir, f"visual_bases_K{best_k}_medoid.pth"))
torch.save(priors_tensor, os.path.join(output_dir, f"priors_K{best_k}.pth"))
np.save(os.path.join(output_dir, "kmeans_labels.base_npy"), labels)

# 為了相容舊程式，如果有需要，可以另外存一個變數
# visual_bases = visual_bases_tensor

--- 步驟 0: 特徵正規化 ---
--- 步驟 1: 自動尋找最佳 K 值 ---
For n_clusters = 8, The average silhouette_score is : 0.7743
For n_clusters = 12, The average silhouette_score is : 0.7597
For n_clusters = 16, The average silhouette_score is : 0.7495
For n_clusters = 20, The average silhouette_score is : 0.7433
For n_clusters = 24, The average silhouette_score is : 0.7366
For n_clusters = 32, The average silhouette_score is : 0.7281
===> 最佳 K 值為: 8 (Score: 0.7743)
--- 步驟 2: 使用 K=8 對「全量數據」進行分群 ---
--- 步驟 3 & 4: 計算 Medoid 與 Prior ---
Cluster 00: Count=2966802, Prior=0.7203, Medoid Original Index=1784048
Cluster 01: Count=187334, Prior=0.0455, Medoid Original Index=2195515
Cluster 02: Count=224256, Prior=0.0544, Medoid Original Index=1352097
Cluster 03: Count=99578, Prior=0.0242, Medoid Original Index=2888797
Cluster 04: Count=115568, Prior=0.0281, Medoid Original Index=173276
Cluster 05: Count=192619, Prior=0.0468, Medoid Original Index=692880
Cluster 06: Count=141434, Prior=0.0343, Medoid Original Index=206

### 拼接圖

In [8]:
import torch
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image
from torchvision import io
import numpy as np
import os
import json
from collections import Counter
from sklearn.metrics.pairwise import euclidean_distances
from PIL import ImageDraw

# ================= 設定區域 =================
# best_k = 8  # 請確保外部已定義此變數
OUTPUT_DIR = "../archive/vlm_knowledge_construction_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATCH_SIZE = 64
ANCHOR_SIZE = 256
GLOBAL_RESIZE = 512

# --- 1. 專家知識注入 (Expert Knowledge Injection) ---
# [新增] 定義每個場域的強度範圍 (Min, Max) - 用於計算 Range
FIELD_INTENSITY_RANGE = {
    "A": (1, 3),   # 黑鯛 (桶養): 弱~中弱 (視覺面積大但能量低)
    "B": (7, 10),  # 海鱺 (箱網): 強~極強 (修正 Min 為 7，區隔出絕對強者)
    "C": (3, 6),   # 石鯛 (室內): 弱~中 (修正範圍，反映面積較小的事實)
    "D": (2, 5),   # 鱸魚 (池養): 弱~中
    "E": (1, 3)    # 午仔 (池養): 微弱~弱
}

# 這是我們剛定義好的「硬事實」，作為 VLM 的先驗知識
# 強度定義 (Baseline): A=3, B=10, C=7, D=4, E=1
FIELD_METADATA_MAPPING = {
    "barrel": {
        "field_id": "A", "intensity_baseline": 3,
        "environment": "outdoor_small_plastic_tank",
        "fish_species": "black_seabream_and_seabass_fry",
        "description": "Small outdoor plastic tank with fish fry. Weak ripples, strong reflection."
    },
    "sea": {
        "field_id": "B", "intensity_baseline": 10,
        "environment": "offshore_cage",
        "fish_species": "cobia_and_pompano",
        "description": "Offshore open sea cage. Large splashes, net cover occlusion possible."
    },
    "LNG": {
        "field_id": "C", "intensity_baseline": 5,
        "environment": "indoor_concrete_pond",
        "fish_species": "hybrid_rock_bream",
        "description": "Indoor concrete pond. Large ripples, artificial lighting reflection."
    },
    "seabass_hmh": {
        "field_id": "D", "intensity_baseline": 4,
        "environment": "onshore_concrete_pond",
        "fish_species": "seabass",
        "description": "Large onshore concrete pond. Mixed ripples and splashes. Machinery in background."
    },
    "noon_jsj": {
        "field_id": "E", "intensity_baseline": 1,
        "environment": "onshore_concrete_pond",
        "fish_species": "fourfinger_threadfin",
        "description": "Onshore pond with Threadfin. Small dot-like splashes. Water wheel interference."
    }
}

def get_expert_metadata(file_path):
    sorted_keys = sorted(FIELD_METADATA_MAPPING.keys(), key=len, reverse=True)
    for key in sorted_keys:
        if key in file_path:
            return FIELD_METADATA_MAPPING[key]
    return {"field_id": "Unknown", "intensity_baseline": 0, "description": "Unknown environment"}

def calculate_intensity_stats(source_counts, total_count):
    """
    計算該 Cluster 的強度統計：
    1. Weighted Average (加權平均)
    2. Min/Max Range (範圍)
    """
    weighted_sum = 0.0
    min_scores = []
    max_scores = []

    for field_id, count in source_counts.items():
        weight = count / total_count

        # 取得該場域的基準強度 (用於平均)
        # 先反查該 field_id 對應的 metadata key
        baseline = 0
        for meta in FIELD_METADATA_MAPPING.values():
            if meta['field_id'] == field_id:
                baseline = meta['intensity_baseline']
                break
        weighted_sum += baseline * weight

        # 取得該場域的強度範圍 (用於 Min/Max)
        # 只有當該場域佔比 > 10% 才納入考量，避免 outlier 影響範圍判斷
        if weight > 0.1 and field_id in FIELD_INTENSITY_RANGE:
            low, high = FIELD_INTENSITY_RANGE[field_id]
            min_scores.append(low)
            max_scores.append(high)

    avg_score = int(round(weighted_sum))

    if not min_scores:
        range_min, range_max = 0, 0
    else:
        # 取聯集範圍: 最低的低標 ~ 最高的標
        range_min = min(min_scores)
        range_max = max(max_scores)

    return avg_score, range_min, range_max

# --- 2. 影像處理輔助函式 (保持不變) ---
def read_and_process_image(path):
    try:
        img = io.read_image(path).float() / 255.0
        if img.shape[0] == 1: img = img.repeat(3, 1, 1)
        elif img.shape[0] == 4: img = img[:3, :, :]
        return img
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

def get_crop_and_bbox(img_tensor, context, crop_size):
    _, H, W = img_tensor.shape
    (y_feat, x_feat) = context['feature_map_coord']
    y_center = int(y_feat * H / 200)
    x_center = int(x_feat * W / 200)
    top = max(0, y_center - crop_size // 2)
    left = max(0, x_center - crop_size // 2)
    if top + crop_size > H: top = H - crop_size
    if left + crop_size > W: left = W - crop_size
    patch = TF.crop(img_tensor, top, left, crop_size, crop_size)
    if patch.shape[1] != crop_size or patch.shape[2] != crop_size:
        patch = TF.resize(patch, [crop_size, crop_size])
    return patch, (left, top, left + crop_size, top + crop_size)

# ================= 主流程 =================

cluster_knowledge_draft = []
print(f"--- 開始執行 Visual Context Construction (K={best_k}) ---")

for k in range(best_k):
    print(f"Processing Cluster {k}...")

    indices = np.where(labels == k)[0]
    cluster_vectors = all_features_norm[indices]

    # 計算 Medoid
    centroid = kmeans.cluster_centers_[k].reshape(1, -1)
    dists_to_centroid = euclidean_distances(cluster_vectors, centroid)
    medoid_local_idx = np.argmin(dists_to_centroid)
    medoid_global_idx = indices[medoid_local_idx]

    # 計算 Variance (Top-25)
    medoid_vector = cluster_vectors[medoid_local_idx].reshape(1, -1)
    dists_to_medoid = euclidean_distances(cluster_vectors, medoid_vector).flatten()
    top_m_local_indices = np.argsort(dists_to_medoid)[:25]
    top_m_global_indices = indices[top_m_local_indices]

    # 統計資料來源
    sample_indices = np.random.choice(indices, min(len(indices), 2000), replace=False)
    source_stats = []
    for idx in sample_indices:
        path = all_context_list[idx]['path']
        meta = get_expert_metadata(path)
        source_stats.append(meta['field_id'])

    source_counts = Counter(source_stats)
    total_count = sum(source_counts.values())
    dist_str = ", ".join([f"{key}({val/total_count:.1%})" for key, val in source_counts.most_common()])

    dominant_field_id = source_counts.most_common(1)[0][0]
    dominant_meta = {}
    for meta in FIELD_METADATA_MAPPING.values():
        if meta['field_id'] == dominant_field_id:
            dominant_meta = meta
            break

    # [新增] 計算進階強度指標 (Avg, Min, Max)
    avg_score, range_min, range_max = calculate_intensity_stats(source_counts, total_count)

    # 生成 Dashboard
    medoid_ctx = all_context_list[medoid_global_idx]
    medoid_img_full = read_and_process_image(medoid_ctx['path'])

    if medoid_img_full is not None:
        anchor_patch, bbox = get_crop_and_bbox(medoid_img_full, medoid_ctx, ANCHOR_SIZE)
        medoid_pil = TF.to_pil_image(medoid_img_full)
        draw = ImageDraw.Draw(medoid_pil)
        draw.rectangle(bbox, outline="red", width=8)
        context_img = TF.to_tensor(medoid_pil.resize((GLOBAL_RESIZE, GLOBAL_RESIZE)))
    else:
        anchor_patch = torch.zeros(3, ANCHOR_SIZE, ANCHOR_SIZE)
        context_img = torch.zeros(3, GLOBAL_RESIZE, GLOBAL_RESIZE)

    variance_patches = []
    for idx in top_m_global_indices:
        ctx = all_context_list[idx]
        img = read_and_process_image(ctx['path'])
        if img is not None:
            p, _ = get_crop_and_bbox(img, ctx, PATCH_SIZE)
            variance_patches.append(p)
        else:
            variance_patches.append(torch.zeros(3, PATCH_SIZE, PATCH_SIZE))

    variance_grid = make_grid(torch.stack(variance_patches), nrow=5, padding=2, normalize=False)
    save_image(variance_grid, os.path.join(OUTPUT_DIR, f"cluster_{k}_grid.png"))
    variance_grid_resized = TF.resize(variance_grid, [GLOBAL_RESIZE, GLOBAL_RESIZE])

    final_dashboard = torch.cat([
        context_img,
        TF.resize(anchor_patch, [GLOBAL_RESIZE, GLOBAL_RESIZE]),
        variance_grid_resized
    ], dim=2)
    dashboard_filename = f"cluster_{k}_dashboard.jpg"
    save_image(final_dashboard, os.path.join(OUTPUT_DIR, dashboard_filename))

    # --- 6. 建構 Knowledge Draft JSON (更新版：包含強度範圍) ---
    cluster_info = {
        "cluster_id": k,
        "visual_dashboard_path": dashboard_filename,
        "expert_metadata": {
            "dominant_field": dominant_field_id,
            "source_distribution": dist_str,
            "environment_type": dominant_meta.get('environment', 'Unknown'),
            "fish_species": dominant_meta.get('fish_species', 'Unknown'),
            # 強度三兄弟: 平均值、最小值、最大值
            "splash_intensity": avg_score,
            "intensity_min": range_min,
            "intensity_max": range_max
        },
        "vlm_tasks": {
            "water_color": "TODO: Select from [dark_blue, greenish, muddy_brown, grey_concrete, black, black_monochrome]",
            "texture_type": "TODO: Select from [smooth_concentric, chaotic_ripples, foamy_white, glassy, striated_noise]",
            "splash_shape": "TODO: Select from [droplets, columnar, ripple, spray, boiling, concentric_ripple, isolated_droplets, wide_spread_splash, chaotic_whitewater]",
            "surface_cover": "TODO: Select from [white_netting_overlay, floating_algae, foam_scum]",
            "interference": "TODO: Select from [aerator_bubbles, bird_or_rat, plastic_pipes, green_net_fencing, distant_cages]",
            "lighting": "TODO: Select from [diffuse, high_glare, shadowed]",
            "container_edge": "TODO: Select from [open_water, plastic_edge, concrete_wall, netting]",
            "description": "TODO: Concise visual description",
        }
    }
    cluster_knowledge_draft.append(cluster_info)

json_path = os.path.join(OUTPUT_DIR, "cluster_knowledge_draft.json")
with open(json_path, "w", encoding='utf-8') as f:
    json.dump(cluster_knowledge_draft, f, indent=4, ensure_ascii=False)

print(f"\n--- 階段一完成 ---")
print(f"1. 視覺儀表板已儲存於: {OUTPUT_DIR}")
print(f"2. 知識草稿 JSON 已儲存於: {json_path}")
print(f"3. 專家強度規則: A(1-3), B(6-10), C(4-8), D(1-5), E(1-4)")

--- 開始執行 Visual Context Construction (K=8) ---
Processing Cluster 0...
Processing Cluster 1...
Processing Cluster 2...
Processing Cluster 3...
Processing Cluster 4...
Processing Cluster 5...
Processing Cluster 6...
Processing Cluster 7...

--- 階段一完成 ---
1. 視覺儀表板已儲存於: vlm_knowledge_construction_v2
2. 知識草稿 JSON 已儲存於: vlm_knowledge_construction_v2\cluster_knowledge_draft.json
3. 專家強度規則: A(1-3), B(6-10), C(4-8), D(1-5), E(1-4)


In [23]:

import glob
from collections import defaultdict

# ================= 設定區域 =================
# 請修改為您的 GT 資料夾路徑
# 假設結構是 root/field_name/labelme_jsons/*.json
GT_ROOT = r"C:\Users\user\PycharmProjects\organized_ripple_4fold"

# 場域代號對照 (根據資料夾名稱)
# 請確認資料夾名稱與代號的對應
FIELD_MAP = {
    "barrel": "A",
    "sea": "B",
    "LNG": "C",
    "seabass_hmh": "D",
    "noon_jsj": "E"
}

# 輸出的 Anchor Mapping 檔案
OUTPUT_ANCHOR_JSON = "vlm_knowledge_construction_v2/anchor_mapping_auto.json"

def calculate_polygon_area(points):
    """
    使用鞋帶公式 (Shoelace Formula) 計算多邊形面積
    """
    area = 0.0
    for i in range(len(points)):
        j = (i + 1) % len(points)
        area += points[i][0] * points[j][1]
        area -= points[j][0] * points[i][1]
    return abs(area) / 2.0

def get_image_splash_area(json_path):
    """
    讀取 LabelMe JSON 並計算所有水花 Label 的總面積
    """
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        total_area = 0.0
        found_splash = False

        for shape in data['shapes']:
            # 假設標籤名稱包含 splash, foam, ripple 等關鍵字
            # 或者如果所有標註都是水花，就直接算
            label = shape['label'].lower()
            # 這裡依據您的標註習慣調整，若所有 mask 都是目標則不需要 if
            if shape['shape_type'] == 'polygon':
                area = calculate_polygon_area(shape['points'])
                total_area += area
                found_splash = True

        return total_area, os.path.basename(data['imagePath'])

    except Exception as e:
        print(f"Error parsing {json_path}: {e}")
        return 0.0, None

def main():
    field_data = defaultdict(list)

    print("正在掃描 LabelMe GT 資料...")

    # 遍歷每個場域資料夾
    for folder_name, field_id in FIELD_MAP.items():
        search_path = os.path.join(GT_ROOT, folder_name, "labelme_jsons", "*.json")
        json_files = glob.glob(search_path)

        print(f"Processing Field {field_id} ({folder_name}): {len(json_files)} files found.")

        for j_path in json_files:
            area, img_name = get_image_splash_area(j_path)
            if img_name and area > 0:
                field_data[field_id].append({
                    "filename": img_name,
                    "area": area
                })

    # 挑選 High/Mid/Low
    final_anchors = {}

    print("\n--- Anchor Selection (Based on GT Area) ---")
    for field_id, items in field_data.items():
        if not items:
            continue

        # 根據面積排序 (由小到大)
        sorted_items = sorted(items, key=lambda x: x['area'])
        n = len(sorted_items)

        # 挑選邏輯：
        # Low: 10% 位置 (避免極端值)
        # Mid: 50% 位置
        # High: 90% 位置

        low_idx = int(n * 0.1)
        mid_idx = int(n * 0.5)
        high_idx = int(n * 0.9)

        anchors = {
            "low": sorted_items[low_idx]['filename'],
            "mid": sorted_items[mid_idx]['filename'],
            "high": sorted_items[high_idx]['filename']
        }

        # 紀錄面積以供參考
        stats = {
            "low_area": sorted_items[low_idx]['area'],
            "mid_area": sorted_items[mid_idx]['area'],
            "high_area": sorted_items[high_idx]['area']
        }

        final_anchors[field_id] = anchors
        print(f"Field {field_id}: Low={anchors['low']} ({stats['low_area']:.0f}), "
              f"Mid={anchors['mid']} ({stats['mid_area']:.0f}), "
              f"High={anchors['high']} ({stats['high_area']:.0f})")

    # 存檔
    os.makedirs(os.path.dirname(OUTPUT_ANCHOR_JSON), exist_ok=True)
    with open(OUTPUT_ANCHOR_JSON, 'w', encoding='utf-8') as f:
        json.dump(final_anchors, f, indent=4)

    print(f"\nAnchor Mapping 已自動生成並儲存至: {OUTPUT_ANCHOR_JSON}")

if __name__ == "__main__":
    main()

正在掃描 LabelMe GT 資料...
Processing Field A (barrel): 500 files found.
Processing Field B (sea): 500 files found.
Processing Field C (LNG): 500 files found.
Processing Field D (seabass_hmh): 500 files found.
Processing Field E (noon_jsj): 97 files found.

--- Anchor Selection (Based on GT Area) ---
Field A: Low=sea829_1_1319.jpg (7440), Mid=barrel0708_1158_1_188.jpg (22155), High=barrel0805_1250_1_554.jpg (61408)
Field B: Low=seaIMG1582_1_30.jpg (12739), Mid=Zed826_1_5.jpg (23788), High=IMG1122_1_107.jpg (108873)
Field C: Low=LNG1119_07_1_205.jpg (4457), Mid=LNG1126_1_403.jpg (16933), High=LNG1112_07_1_509.jpg (39241)
Field D: Low=2024-04-09-06-57-01.png (1626), Mid=2024-04-09-06-40-31.png (7086), High=output-2023-08-07-07-45-42-00000066.png (53647)
Field E: Low=output-2023-07-27-08-03-57-00000007.png (756), Mid=output-2023-10-19-05-38-10-00000009.png (10930), High=output-2023-10-16-16-02-13-00000069.png (27916)

Anchor Mapping 已自動生成並儲存至: vlm_knowledge_construction_v2/anchor_mapping_auto.

In [9]:

import os

# ================= 設定區域 =================
INPUT_DIR = "../archive/vlm_knowledge_construction_v2"
INPUT_JSON = os.path.join(INPUT_DIR, "cluster_knowledge_draft.json")

def main():
    if not os.path.exists(INPUT_JSON):
        print(f"錯誤: 找不到 {INPUT_JSON}")
        return

    with open(INPUT_JSON, "r", encoding='utf-8') as f:
        data = json.load(f)

    print("========================================================")
    print(f" Gemini/GPT-4o 專用 Prompt 生成器 (視覺紋理分析版)")
    print("========================================================\n")

    for entry in data:
        c_id = entry['cluster_id']
        img_path = entry['visual_dashboard_path']
        meta = entry['expert_metadata']

        # 這裡會讀取我們在 Dashboard Generator 算好的統計強度
        avg_intensity = meta.get('splash_intensity', 'Unknown')
        min_int = meta.get('intensity_min', 'Unknown')
        max_int = meta.get('intensity_max', 'Unknown')

        # 組合強度描述字串，給 VLM 當參考
        intensity_context = f"Average: {avg_intensity}/10 (Range: {min_int}-{max_int})"

        print(f"### Cluster {c_id} (請上傳: {img_path}) ###")
        print("-" * 20 + " 複製下方文字 " + "-" * 20)

        # CoT Prompt (強度作為已知條件)
        prompt = f"""
I am analyzing aquaculture water splash patterns. Act as a Computer Vision Expert.

I have provided a "Visual Dashboard" composed of:
1. **LEFT (Context):** A global view.
2. **CENTER (Anchor):** A close-up prototype.
3. **RIGHT (Variance):** 25 random samples from this cluster.

---
[HARD CONSTRAINTS] (Expert Metadata - These are GROUND TRUTHS)
* Environment: {meta.get('environment_type')}
* Source Field: {meta.get('dominant_field')} ({meta.get('source_distribution')})
* Fish Species: {meta.get('fish_species')}
* **Calculated Intensity Profile: {intensity_context}**
  (Note: This intensity is statistically derived from field data. Do NOT guess the score, but describe features matching this intensity.)
---

[DOMAIN KNOWLEDGE RULES]
1. **Color & Environment:**
   - **Offshore/Sea:** Dark blue, Black.
   - **Onshore Ponds:** Greenish (Algae), Muddy Brown, Grey (Concrete).

2. **Visual Texture Mapping:**
   - **High Intensity (7-10):** Look for "Boiling", "Chaotic Whitewater", "Heavy Foam". Common in Cobia feeding.
   - **Medium Intensity (4-6):** Look for "Droplets", "Spray", "Distinct Splashes". Common in Pompano/Threadfin.
   - **Low Intensity (1-3):** Look for "Ripples", "Glassy Surface", "Small Disturbances".

3. **Interference Awareness:**
   - **Nets:** White grid overlays.
   - **Aerators:** Mechanical bubbles/foam.
   - **Glares:** High contrast reflections (do not mistake for white foam).

---
[TASK: VISUAL TEXTURE ANALYSIS]
Your goal is to generate a structured description of the visual texture.

**Step 1: Analyze Texture & Shape**
Observe the Anchor and Variance images. Describe the water surface state using the vocabulary below.

**Step 2: Check for Interference**
Identify non-splash elements (nets, pipes, birds) that might be visually dominant.

**Step 3: Generate Structured JSON**
Provide your analysis for Step 1 & 2 briefly, followed by the JSON object.

```json
{{
    "water_color": "Select one: [dark_blue, greenish, muddy_brown, grey_concrete, black, black_monochrome]",
    "texture_type": "Select one: [smooth_concentric, chaotic_ripples, foamy_white, glassy, striated_noise, churning_oil]",
    "splash_shape": "Select one: [droplets, columnar, ripple, spray, boiling, concentric_ripple, isolated_droplets, wide_spread_splash, chaotic_whitewater, faint_disturbance]",
    "surface_cover": "Select one: [none, white_netting_overlay, floating_algae, foam_scum]",
    "interference": "Select one: [none, aerator_bubbles, bird_or_rat, plastic_pipes, green_net_fencing, distant_cages, reflections]",
    "lighting": "Select one: [diffuse, high_glare, shadowed]",
    "container_edge": "Select one: [open_water, plastic_edge, concrete_wall, netting]",
    "description": "A concise sentence describing the visual appearance, explicitly mentioning texture and dynamics (e.g., 'Chaotic boiling white foam with dark blue water background')."
}}"""
        print(prompt.strip())
        print("-" * 55)
        print("\n\n")

if __name__ == "__main__":
    main()

 Gemini/GPT-4o 專用 Prompt 生成器 (視覺紋理分析版)

### Cluster 0 (請上傳: cluster_0_dashboard.jpg) ###
-------------------- 複製下方文字 --------------------
I am analyzing aquaculture water splash patterns. Act as a Computer Vision Expert.

I have provided a "Visual Dashboard" composed of:
1. **LEFT (Context):** A global view.
2. **CENTER (Anchor):** A close-up prototype.
3. **RIGHT (Variance):** 25 random samples from this cluster.

---
[HARD CONSTRAINTS] (Expert Metadata - These are GROUND TRUTHS)
* Environment: offshore_cage
* Source Field: B (B(45.5%), A(29.8%), C(19.3%), D(5.5%))
* Fish Species: cobia_and_pompano
* **Calculated Intensity Profile: Average: 7/10 (Range: 1-10)**
  (Note: This intensity is statistically derived from field data. Do NOT guess the score, but describe features matching this intensity.)
---

[DOMAIN KNOWLEDGE RULES]
1. **Color & Environment:**
   - **Offshore/Sea:** Dark blue, Black.
   - **Onshore Ponds:** Greenish (Algae), Muddy Brown, Grey (Concrete).

2. **Visual Texture

In [13]:


def generate_final_knowledge_graph_json(input_path, output_path):
    """
    讀取 cluster_knowledge_filled.json，
    提取 expert_metadata 與 vlm_tasks 的關鍵欄位，
    整合至新的 graph_rag 欄位中，並輸出為 cluster_knowledge_final.json。
    """

    # 1. 檢查檔案是否存在
    if not os.path.exists(input_path):
        print(f"錯誤: 找不到輸入檔案 {input_path}")
        return

    # 2. 讀取 JSON 資料
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    print(f"成功讀取 {len(data)} 筆 Cluster 資料，開始處理...")

    # 3. 遍歷並處理每一筆 Cluster
    for cluster in data:
        # 取得來源字典 (若無則給空字典避免報錯)
        expert = cluster.get("expert_metadata", {})
        vlm = cluster.get("vlm_tasks", {})

        # 4. 建構 graph_rag 字典 (扁平化屬性)
        graph_rag_data = {
            # 來自 expert_metadata 的數據 (數值與統計類)
            "dominant_field": expert.get("dominant_field", "Unknown"),
            "environment_type": expert.get("environment_type", "Unknown"),
            "fish_species": expert.get("fish_species", "Unknown"),
            "splash_intensity": expert.get("splash_intensity", 0),
            "intensity_min": expert.get("intensity_min", 0),
            "intensity_max": expert.get("intensity_max", 0),
            "source_distribution": expert.get("source_distribution", "Unknown"),

            # 來自 vlm_tasks 的數據 (視覺描述類)
            "water_color": vlm.get("water_color", "Unknown"),
            "texture_type": vlm.get("texture_type", "Unknown"),
            "splash_shape": vlm.get("splash_shape", "Unknown"),
            "surface_cover": vlm.get("surface_cover", "None"),
            "interference": vlm.get("interference", "None"),
            "lighting": vlm.get("lighting", "Unknown"),
            "container_edge": vlm.get("container_edge", "Unknown"),
            "description": vlm.get("description", "")
        }

        # 5. 將整理好的 graph_rag 插入 Cluster 物件
        cluster["graph_rag"] = graph_rag_data

    # 6. 寫入新檔案
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"處理完成！新檔案已儲存至: {output_path}")
    print("您現在可以直接讀取 'graph_rag' 欄位來生成 Neo4j 節點與關係。")

# --- 執行設定 ---
input_json = 'vlm_knowledge_construction_v2\cluster_knowledge_filled.json'  # 您的來源檔案
output_json = 'vlm_knowledge_construction_v2\cluster_knowledge_final.json'  # 您的目標檔案

generate_final_knowledge_graph_json(input_json, output_json)

成功讀取 8 筆 Cluster 資料，開始處理...
處理完成！新檔案已儲存至: vlm_knowledge_construction_v2\cluster_knowledge_final.json
您現在可以直接讀取 'graph_rag' 欄位來生成 Neo4j 節點與關係。


In [14]:
import torch
import numpy as np
# 雖然不需要重跑 KMeans，但保留 import 以防未來需要

# ================= 設定區域 =================
# 1. 原始特徵與資料路徑
FEATURE_PATH = "../archive/prompt/all_features_tensor.pth"
CONTEXT_PATH = "../archive/prompt/all_context_list.pth"

# 2. [修改] Labels 讀取路徑 (強制讀取此檔)
LABELS_INPUT_PATH = "../archive/prompt/kmeans_labels.npy"

# 3. 最終輸出的全域索引表
OUTPUT_MAPPING = "vlm_knowledge_construction_v2/patch_to_cluster_mapping.json"

# [重要] 只是為了紀錄與檢查，不會用於重跑
K_CLUSTERS = 8

def main():
    print("--- 步驟 1: 載入原始資料 (Context) ---")
    if not os.path.exists(CONTEXT_PATH):
        print(f"錯誤：找不到 {CONTEXT_PATH}")
        return

    # 載入 Context (這裡面已經有 'intensity' 了)
    # 我們這裡其實不需要載入 Feature Tensor 了，因為分群已經做完
    # 但為了確保數據筆數一致，我們載入 Context 即可
    all_context = torch.load(CONTEXT_PATH)
    print(f"Context 載入完成，共 {len(all_context)} 筆")

    # ================= 讀取 Labels =================
    print(f"\n--- 步驟 2: 讀取 K-means Labels (K={K_CLUSTERS}) ---")

    if not os.path.exists(LABELS_INPUT_PATH):
        print(f"❌ 致命錯誤：找不到 Labels 檔案: {LABELS_INPUT_PATH}")
        print("請先執行上一步驟的 K-Means 分群腳本來生成此檔案！")
        return

    # [修改核心] 直接讀取，不再重跑
    labels = np.load(LABELS_INPUT_PATH)
    print(f"✅ 成功讀取 Labels，共 {labels.shape[0]} 筆分群結果")

    # 簡單防呆檢查：確保 Context 數量跟 Labels 數量一樣
    if len(all_context) != labels.shape[0]:
        print(f"⚠️ 警告：Context 筆數 ({len(all_context)}) 與 Labels 筆數 ({labels.shape[0]}) 不一致！")
        print("這可能導致 Mapping 錯亂，請檢查檔案版本是否匹配。")
        return

    # ================= 建立 Mapping =================
    print("\n--- 步驟 3: 建立 Patch Mapping (包含強度資訊) ---")

    full_mapping = []

    # zip 把 context (來源) 和 label (分群結果) 對在一起
    for idx, (ctx, cluster_id) in enumerate(zip(all_context, labels)):
        # 取得檔名 (不含路徑)
        fname = os.path.basename(ctx['path'])

        # 基本資訊
        patch_info = {
            "patch_id": idx,                # Graph RAG 檢索回來的 ID
            "cluster_id": int(cluster_id),  # 連結 Knowledge Graph 的 ID
            "filename": fname,
            "original_path": ctx['path'],

            # [關鍵] 將強度資訊轉存過來，這是未來做 Top 25% 排序的依據
            # 使用 .get(..., 0) 防呆，若無該欄位則補 0
            "intensity": ctx.get('intensity', 0)
        }

        full_mapping.append(patch_info)

    # 儲存最終 Mapping
    os.makedirs(os.path.dirname(OUTPUT_MAPPING), exist_ok=True)
    with open(OUTPUT_MAPPING, "w", encoding='utf-8') as f:
        json.dump(full_mapping, f, indent=4)

    print(f"\n全域索引表已成功建立：{OUTPUT_MAPPING}")
    print(f"總筆數: {len(full_mapping)}")
    print("現在您可以進行下一步：建立 Neo4j 圖譜。")

if __name__ == "__main__":
    main()

--- 步驟 1: 載入原始資料 (Context) ---


C:\Users\user\AppData\Local\Temp\ipykernel_33288\2636997779.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  all_context = torch.load(CONTEXT_PATH)


Context 載入完成，共 4118667 筆

--- 步驟 2: 讀取 K-means Labels (K=8) ---
✅ 成功讀取 Labels，共 4118667 筆分群結果

--- 步驟 3: 建立 Patch Mapping (包含強度資訊) ---

全域索引表已成功建立：vlm_knowledge_construction_v2/patch_to_cluster_mapping.json
總筆數: 4118667
現在您可以進行下一步：建立 Neo4j 圖譜。


In [4]:
import json
import networkx as nx
import os
import itertools
import re

# ================= 設定區域 =================
INPUT_JSON = "vlm_knowledge_construction_v2/cluster_knowledge_final.json"
GRAPH_OUTPUT_PATH = "../archive/vlm_knowledge_construction_v2/knowledge_graph.gml"

# [C. 強度邏輯] 魚種強度階級表 (數值越大越強) - 這是「基準值 (Baseline)」
SPECIES_RANK = {
    "cobia_and_pompano": 10,              # Field B
    "hybrid_rock_bream": 5,               # Field C
    "seabass": 4,                         # Field D
    "black_seabream_and_seabass_fry": 3,  # Field A
    "fourfinger_threadfin": 1             # Field E
}

# [反查表] 用於將 Source Distribution 中的代號 (A~E) 轉回魚種名稱
FIELD_TO_SPECIES = {
    "A": "black_seabream_and_seabass_fry",
    "B": "cobia_and_pompano",
    "C": "hybrid_rock_bream",
    "D": "seabass",
    "E": "fourfinger_threadfin"
}

def parse_distribution(dist_str):
    """
    [A. 解析混合成分]
    將字串 "D(58.9%), E(39.4%), B(0.8%)" 解析為字典
    Returns: {'D': 0.589, 'E': 0.394, 'B': 0.008}
    """
    if not dist_str:
        return {}
    # 正則表達式抓取：大寫字母 + 括號內的數字
    matches = re.findall(r'([A-E])\(([\d\.]+)\%\)', dist_str)
    return {field: float(val)/100 for field, val in matches}

def create_exhaustive_knowledge_graph(json_data):
    G = nx.DiGraph()
    print("正在建構雙軌邏輯知識圖譜 (Dual-Track Graph Construction)...")

    existing_species_nodes = set()

    for entry in json_data:
        rag_data = entry.get('graph_rag', {})
        c_id = f"Cluster_{entry['cluster_id']}"

        # ================= 1. Cluster Node (觀測值) =================
        # 這裡紀錄的是「觀測到的強度」，例如 Cluster 4 雖然含午仔魚，但整體強度是 3
        G.add_node(c_id,
                   type="Cluster",
                   label=f"Cluster {entry['cluster_id']}",
                   description=rag_data.get('description', ''),
                   avg_intensity=rag_data.get('splash_intensity', 0),
                   min_intensity=rag_data.get('intensity_min', 0),
                   max_intensity=rag_data.get('intensity_max', 0)
                   )

        # ================= 2. 快速通道 (Dominant Path) =================
        # 這是舊邏輯，用於快速篩選場域類型
        field_id = rag_data.get('dominant_field', 'Unknown')
        env_type = rag_data.get('environment_type', 'Unknown')

        field_node = f"Field_{field_id}"
        G.add_node(field_node, type="Field", label=f"Field {field_id}")
        # 標記這是主導場域關係
        G.add_edge(c_id, field_node, relation="ORIGINATES_FROM", type="dominant")

        if env_type and env_type != "Unknown":
            env_node = f"Env_{env_type}"
            G.add_node(env_node, type="Environment", label=env_type)
            G.add_edge(field_node, env_node, relation="IS_TYPE")

        # ================= 3. 細節通道 (Distribution & Weighted Edges) =================
        # [核心修改] 解析 source_distribution 來建立物種關係
        dist_str = rag_data.get('source_distribution', '')
        ratios = parse_distribution(dist_str) # e.g., {'D': 0.589, 'E': 0.394}

        VALID_THRESHOLD = 0.10  # 門檻：10%

        # 如果解析失敗(舊資料防呆)，則回退使用 fish_species
        if not ratios:
            fallback_species = rag_data.get('fish_species')
            if fallback_species:
                # 假設主導物種佔 100% (權宜之計)
                field_code = next((k for k, v in FIELD_TO_SPECIES.items() if v == fallback_species), 'Unknown')
                if field_code != 'Unknown':
                    ratios = {field_code: 1.0}

        # 遍歷成分，建立關係
        for field_code, ratio in ratios.items():
            if ratio >= VALID_THRESHOLD:
                fish_name = FIELD_TO_SPECIES.get(field_code)

                if fish_name:
                    sp_key = fish_name
                    sp_node = f"Species_{sp_key}"
                    baseline = SPECIES_RANK.get(sp_key, 0)

                    # 3.1 建立物種節點 (若不存在)
                    if sp_node not in existing_species_nodes:
                        G.add_node(sp_node, type="Species", label=sp_key, baseline_intensity=baseline)
                        existing_species_nodes.add(sp_key)

                        # 建立 Species -> Baseline Intensity (理論值)
                        base_int_node = f"Intensity_{baseline}"
                        G.add_node(base_int_node, type="Intensity", label=str(baseline), level=baseline)
                        G.add_edge(sp_node, base_int_node, relation="HAS_BASELINE_INTENSITY")

                    # [B. 建立加權關係] Cluster -> Species
                    # 這行讓午仔魚 (E) 即使在 Cluster 4 (D主導) 也能被連結
                    G.add_edge(c_id, sp_node, relation="CONTAINS_SPECIES", ratio=ratio)

        # ================= 4. Intensity Node (Cluster Level) =================
        avg_int = rag_data.get('splash_intensity', 0)
        int_node = f"Intensity_{avg_int}"
        G.add_node(int_node, type="Intensity", label=str(avg_int), level=avg_int)
        G.add_edge(c_id, int_node, relation="HAS_INTENSITY")

        # ================= 5. Visual Features (不變) =================
        attributes = {
            'water_color': 'Color',
            'texture_type': 'Texture',
            'splash_shape': 'Shape',
            'surface_cover': 'Cover',
            'interference': 'Interference',
            'lighting': 'Lighting',
            'container_edge': 'EdgeMaterial'
        }

        relation_map = {
            'water_color': 'HAS_COLOR',
            'texture_type': 'HAS_TEXTURE',
            'splash_shape': 'HAS_SHAPE',
            'surface_cover': 'COVERED_BY',
            'interference': 'INTERFERED_BY',
            'lighting': 'LIT_BY',
            'container_edge': 'BOUNDED_BY'
        }

        for key, node_type in attributes.items():
            val = rag_data.get(key)
            if val and val != 'none':
                feat_node = f"{node_type}_{val}"
                G.add_node(feat_node, type=node_type, label=val)
                G.add_edge(c_id, feat_node, relation=relation_map[key])

    # ================= [C. 強度邏輯] 建立物種間關係 =================
    print("正在建立魚種強度階級關係 (STRONGER / WEAKER / SIMILAR)...")

    species_list = list(existing_species_nodes)

    # 排列組合兩兩比對
    for sp_a, sp_b in itertools.permutations(species_list, 2):
        rank_a = SPECIES_RANK.get(sp_a, 0)
        rank_b = SPECIES_RANK.get(sp_b, 0)

        node_a = f"Species_{sp_a}"
        node_b = f"Species_{sp_b}"

        # if rank_a > rank_b:
        #     G.add_edge(node_a, node_b, relation="STRONGER_THAN")
        # elif rank_a < rank_b:
        #     G.add_edge(node_a, node_b, relation="WEAKER_THAN")
        # elif rank_a == rank_b:
        #     G.add_edge(node_a, node_b, relation="SIMILAR_INTENSITY_TO")

        diff = rank_a - rank_b

        # 建立強弱關係 (差異超過 1 才算強弱)
        if diff > 1:
            G.add_edge(node_a, node_b, relation="STRONGER_THAN")
        elif diff < -1:
            G.add_edge(node_a, node_b, relation="WEAKER_THAN")

        # 建立相似關係 (差異在 1 以內都算相似)
        if abs(diff) <= 1:
            G.add_edge(node_a, node_b, relation="SIMILAR_INTENSITY_TO")

    print(f"圖譜建構完成！")
    print(f"節點總數: {G.number_of_nodes()}")
    print(f"邊總數: {G.number_of_edges()}")
    return G

def main():
    if not os.path.exists(INPUT_JSON):
        print(f"錯誤: 找不到 {INPUT_JSON}")
        return

    with open(INPUT_JSON, "r", encoding='utf-8') as f:
        data = json.load(f)

    kg = create_exhaustive_knowledge_graph(data)

    # 確保輸出目錄存在
    os.makedirs(os.path.dirname(GRAPH_OUTPUT_PATH), exist_ok=True)

    nx.write_gml(kg, GRAPH_OUTPUT_PATH)
    print(f"GML 檔案已儲存至: {GRAPH_OUTPUT_PATH}")

if __name__ == "__main__":
    main()

正在建構雙軌邏輯知識圖譜 (Dual-Track Graph Construction)...
正在建立魚種強度階級關係 (STRONGER / WEAKER / SIMILAR)...
圖譜建構完成！
節點總數: 45
邊總數: 120
GML 檔案已儲存至: vlm_knowledge_construction_v2/knowledge_graph.gml
